In [ ]:
BACKBONE = "ViT-B-32"      # e.g. "RN50", "ViT-B-32", "ViT-L-14"
PRESET   = "cpu-fast"      # "gpu", "cpu-fast", "cpu-tiny"
MAX_SAMP = 10_000          
BATCH    = None            

import json, time, numpy as np, pandas as pd
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import torch, open_clip
from open_clip import tokenize
from torchvision import transforms as T

PRESETS = {
    "gpu":      dict(device="cuda", batch=128, max_samples=None),
    "cpu-fast": dict(device="cpu",  batch=64,  max_samples=50_000),
    "cpu-tiny": dict(device="cpu",  batch=64,  max_samples=10_000),
}
cfg = PRESETS[PRESET].copy()
cfg.update(model=BACKBONE,
           pretrained="laion2b_s34b_b79k",
           batch=BATCH or cfg["batch"],
           max_samples=MAX_SAMP or cfg["max_samples"],
           device="cuda" if PRESET=="gpu" and torch.cuda.is_available() else "cpu")
print("Config →", cfg)

PARQUET = Path(r"C:/Users/steph/OneDrive/Desktop/data/metadata.parquet")
meta = pd.read_parquet(PARQUET)
meta = meta[meta["split"] == "train"].reset_index(drop=True)
if cfg["max_samples"] and cfg["max_samples"] < len(meta):
    meta = meta.sample(cfg["max_samples"], random_state=0).reset_index(drop=True)
print(f"Subset size: {len(meta):,}")

# Dataset & DataLoader
DATA_ROOT = Path(r"C:/Users/steph/OneDrive/Desktop/data")
def resolve_path(r):
    if "image_path" in r and isinstance(r["image_path"], (str, Path)):
        return Path(r["image_path"])
    if r.domain == "coco":
        return DATA_ROOT/"coco"/"train2017"/f"{int(r.image_id):012d}.jpg"
    if r.domain == "sd":
        return DATA_ROOT/"SD"/"images"/r.image_id
    return DATA_ROOT/"flickr30k"/"flickr30k_images"/r.image_id

transform = T.Compose([
    T.Resize(224, antialias=True), T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize((0.48145466,0.4578275,0.40821073),
                (0.26862954,0.26130258,0.27577711)),
])
class RetrievalDS(torch.utils.data.Dataset):
    def __len__(self): return len(meta)
    def __getitem__(self, idx):
        row = meta.iloc[idx]
        try:
            img = transform(Image.open(resolve_path(row)).convert("RGB"))
        except Exception:
            img = torch.zeros(3, 224, 224)
        return {"image": img, "text": row.caption, "id": idx}

loader = torch.utils.data.DataLoader(
    RetrievalDS(), batch_size=cfg["batch"], shuffle=False,
    num_workers=0,  
    pin_memory=(cfg["device"] == "cuda")
)
print("DataLoader ready:", len(loader), "batches")

# Load backbone
t0 = time.time()
model, _, _ = open_clip.create_model_and_transforms(
    cfg["model"], pretrained=cfg["pretrained"], device=cfg["device"])
model.eval(); TOK = tokenize
param_cnt = int(sum(p.numel() for p in model.parameters()))
print(f"{cfg['model']} | {param_cnt/1e6:.1f} M params | load {time.time()-t0:.1f}s")

# Embed
img_chunks, txt_chunks = [], []
t0 = time.time()
with torch.no_grad():
    for batch in tqdm(loader, desc="Embedding"):
        imgs = batch["image"].to(cfg["device"], non_blocking=True)
        caps = TOK(batch["text"]).to(cfg["device"], non_blocking=True)
        ie = model.encode_image(imgs); ie /= ie.norm(dim=-1, keepdim=True)
        te = model.encode_text(caps); te /= te.norm(dim=-1, keepdim=True)
        img_chunks.append(ie.cpu()); txt_chunks.append(te.cpu())
embed_secs = round(time.time()-t0, 1)
print(f"Embedding {len(meta):,} pairs in {embed_secs/60:.1f} min")

IMG_EMB = torch.cat(img_chunks).numpy().astype("float16")
TXT_EMB = torch.cat(txt_chunks).numpy().astype("float16")

# Recall@K
SIM = TXT_EMB @ IMG_EMB.T
def recall_at_k(mat, k):
    topk = np.argpartition(-mat, k-1, axis=1)[:, :k]
    return 100*np.mean(np.any(topk == np.arange(len(mat))[:,None], axis=1))
metrics = {f"R@{k}": round(recall_at_k(SIM, k), 2) for k in (1,5,10)}
print("Recall metrics:", metrics)

RUN_DIR = Path("experiments") / f"{cfg['model']}_{datetime.now():%Y%m%d_%H%M%S}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
np.save(RUN_DIR/"img_embs.npy", IMG_EMB)
np.save(RUN_DIR/"txt_embs.npy", TXT_EMB)
json.dump(metrics, open(RUN_DIR/"metrics.json","w"), indent=2)
json.dump(cfg | {"embed_secs": embed_secs, "param_count": param_cnt},
          open(RUN_DIR/"config.json","w"), indent=2)
print("Saved artefacts →", RUN_DIR)


Config → {'device': 'cpu', 'batch': 64, 'max_samples': 10000, 'model': 'ViT-B-32', 'pretrained': 'laion2b_s34b_b79k'}
Subset size: 10,000
DataLoader ready: 157 batches
ViT-B-32 | 151.3 M params | load 4.5s


Embedding: 100%|█████████████████████████████████████████████████████████████████████| 157/157 [21:20<00:00,  8.16s/it]


Embedding 10,000 pairs in 21.3 min
Recall metrics: {'R@1': 43.09, 'R@5': 66.63, 'R@10': 75.29}
Saved artefacts → experiments\ViT-B-32_20250624_170054
